# Plotagem de dados da análise

Este notebook gera um gráfico para cada arquivo CSV em `amostras/`, possibilitando a análise do tempo de execução de cada algoritmo (em milissegundos) com base em valores variáveis de `n` (de 1 a 100.000 para algoritmos de busca e de 1 a 50.000 para algoritmos de ordenação).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
arquivos = [
    {
        "arquivo": "amostras/busca_sequencial.csv",
        "label": "Busca Sequencial",
        "complexidade": "n",
    },
    {
        "arquivo": "amostras/busca_binaria.csv",
        "label": "Busca Binária",
        "complexidade": "logn",
    },
    {
        "arquivo": "amostras/insertion_sort.csv",
        "label": "Insertion Sort",
        "complexidade": "n2",
    },
    {
        "arquivo": "amostras/selection_sort.csv",
        "label": "Selection Sort",
        "complexidade": "n2",
    },
]

In [ ]:
def complexidade_func(x, tipo):
    if tipo == "n":
        return x
    elif tipo == "n2":
        return x**2
    elif tipo == "nlogn":
        return x * np.log2(x)
    elif tipo == "logn":
        return np.log2(x)
    elif tipo == "1":
        return np.ones_like(x)
    else:
        raise ValueError(f"Complexidade desconhecida: {tipo}")

In [ ]:
for cfg in arquivos:
    df = pd.read_csv(cfg["arquivo"])

    Q1 = df.iloc[:, 1].quantile(0.25)
    Q3 = df.iloc[:, 1].quantile(0.75)
    IQR = Q3 - Q1

    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    df = df[(df.iloc[:, 1] >= limite_inf) & (df.iloc[:, 1] <= limite_sup)]
    df = df.groupby(df.columns[0]).median().reset_index()

    x = df.iloc[:, 0].values  # tamanho (n)
    y = df.iloc[:, 1].values  # tempo (ms)

    # Curva teórica
    f = complexidade_func(x, cfg["complexidade"])

    if cfg["complexidade"] in ["n", "logn"]:
        # mínimos quadrados
        c = np.dot(y, f) / np.dot(f, f)
        y_teorico = c * f
    else:
        # max scaling
        y_teorico = f * (y.max() / f.max())

    plt.figure()
    plt.plot(x, y, label="Tempo medido (mediana)")
    plt.plot(x, y_teorico, linestyle="--", label=f"O({cfg['complexidade']})")

    plt.xlabel("Tamanho da entrada (n)")
    plt.ylabel("Tempo de execução")
    plt.title(cfg["label"])
    plt.legend()
    plt.grid()

    plt.show()